# Synthetic Data Generator
### Step 7 : Kafka Producer Adapter

In [2]:
import os

from utils.database.postgres import (
    get_engine_from_env, 
    read_sql_dataframe,
)


from utils.synthetic.pipeline.kafka_producer_adapter import (
    run_send_queue_producer_once, 
    run_send_queue_producer_loop, 
    build_sensor_message_payload, 
    json_dumps_safe,
)

In [3]:
SCHEMA = os.getenv("CAPSTONE_SCHEMA", "capstone")

DATASET_ID = os.getenv("SYNTHETIC_DATASET_ID", "pump_synthetic_v1")
RUN_ID = os.getenv("SYNTHETIC_RUN_ID", "synthetic_run_001")

SIMULATION_TABLE_NAME = "simulation_state_control"
SEND_QUEUE_TABLE_NAME = "synthetic_sensor_messages_send_queue"

PRODUCER_TOPIC = os.getenv(
    "SYNTHETIC_KAFKA_TOPIC",
    "pump.telemetry.synthetic",
)

PRODUCER_WORKER_ID = os.getenv(
    "SYNTHETIC_PRODUCER_WORKER_ID",
    "producer_worker_001",
)

CLIENT_ID = os.getenv(
    "SYNTHETIC_PRODUCER_CLIENT_ID",
    "pump-telemetry-producer",
)

OBSERVATION_BATCH_SIZE = int(os.getenv("SYNTHETIC_OBSERVATION_BATCH_SIZE", "500"))
PRODUCER_POLL_SECONDS = float(os.getenv("SYNTHETIC_PRODUCER_POLL_SECONDS", "0.0"))
PRODUCER_MAX_SEND_ATTEMPTS = int(os.getenv("SYNTHETIC_PRODUCER_MAX_SEND_ATTEMPTS", "3"))

FLUSH_TIMEOUT_SECONDS = float(os.getenv("SYNTHETIC_PRODUCER_FLUSH_TIMEOUT_SECONDS", "30.0"))


RUN_PRODUCER_LOOP_FLAG = True

RUN_ONE_BATCH_PRODUCER_SMOKE_TEST_FLAG = False

MAX_BATCHES_RAW_LIMIT = (
    os.getenv("SYNTHETIC_PRODUCER_MAX_BATCHES_LIMIT")
    or os.getenv("PRODUCER_MAX_BATCHES_LIMIT")
    or "1"
).strip()

if MAX_BATCHES_RAW_LIMIT.lower() in {"none", "null", "all", ""}:
    MAX_BATCHES = None
else:
    MAX_BATCHES = int(MAX_BATCHES_RAW_LIMIT)


STOP_ON_FAILURE_FLAG = True




print("Producer Adapter config")
print("schema:", SCHEMA)
print("dataset id:", DATASET_ID)
print("run id:", RUN_ID)
print("send queue table:", SEND_QUEUE_TABLE_NAME)
print("producer topic:", PRODUCER_TOPIC)
print("observation batch size:", OBSERVATION_BATCH_SIZE)
print("client id:", CLIENT_ID)
print("flush timeout seconds:", FLUSH_TIMEOUT_SECONDS)
print("run producer loop:", RUN_PRODUCER_LOOP_FLAG)
print("producer max batches limit raw:", MAX_BATCHES_RAW_LIMIT)
print("producer max batches:", MAX_BATCHES)
print("stop on failure:", STOP_ON_FAILURE_FLAG)



Producer Adapter config
schema: capstone
dataset id: pump_synthetic_v1
run id: synthetic_run_001
send queue table: synthetic_sensor_messages_send_queue
producer topic: pump.telemetry.synthetic
observation batch size: 500
client id: pump-telemetry-producer
flush timeout seconds: 30.0
run producer loop: True
producer max batches limit raw: None
producer max batches: None
stop on failure: True


---

In [4]:
engine = get_engine_from_env()

---

In [5]:
NUMBER_OF_SENSORS = int(
    read_sql_dataframe(
        engine,
        f"""
        SELECT COUNT(DISTINCT sensor_index) AS sensor_count
        FROM "{SCHEMA}"."{SEND_QUEUE_TABLE_NAME}"
        WHERE dataset_id = :dataset_id
          AND run_id = :run_id
        """,
        params={
            "dataset_id": DATASET_ID,
            "run_id": RUN_ID,
        },
    ).loc[0, "sensor_count"]
)

PRODUCER_BATCH_SIZE = OBSERVATION_BATCH_SIZE * NUMBER_OF_SENSORS

print("Derived producer batch sizing")
print("number of sensors:", NUMBER_OF_SENSORS)
print("observation batch size:", OBSERVATION_BATCH_SIZE)
print("producer batch size:", PRODUCER_BATCH_SIZE)

Derived producer batch sizing
number of sensors: 52
observation batch size: 500
producer batch size: 26000


----

In [6]:
pre_send_queue_status_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        queue_status,
        COUNT(*) AS row_count,
        COUNT(*) FILTER (WHERE claim_token IS NULL) AS null_claim_token_count,
        COUNT(*) FILTER (WHERE claim_token IS NOT NULL) AS populated_claim_token_count,
        COUNT(*) FILTER (WHERE producer_sent_at IS NULL) AS unsent_count,
        COUNT(*) FILTER (WHERE producer_sent_at IS NOT NULL) AS sent_count
    FROM "{SCHEMA}"."{SEND_QUEUE_TABLE_NAME}"
    WHERE dataset_id = :dataset_id
      AND run_id = :run_id
    GROUP BY queue_status
    ORDER BY row_count DESC
    """,
    params={
        "dataset_id": DATASET_ID,
        "run_id": RUN_ID,
    },
)

display(pre_send_queue_status_dataframe)

,queue_status,row_count,null_claim_token_count,populated_claim_token_count,unsent_count,sent_count
0,pending,11674000,11674000,0,11674000,0
1,sent,26000,0,26000,0,26000


----

In [7]:
preview_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT *
    FROM "{SCHEMA}"."{SEND_QUEUE_TABLE_NAME}"
    WHERE dataset_id = :dataset_id
      AND run_id = :run_id
      AND queue_status = 'pending'
    ORDER BY observation_index, message_sequence_index, sensor_index
    LIMIT 1
    """,
    params={
        "dataset_id": DATASET_ID,
        "run_id": RUN_ID,
    },
)

if preview_dataframe.empty:
    print("No pending rows available for payload preview.")
else:
    payload = build_sensor_message_payload(preview_dataframe.iloc[0].to_dict())
    print(json_dumps_safe(payload))

{"message_key": "pump_synthetic_v1|synthetic_run_001|pump_asset_001|501|0", "dataset_id": "pump_synthetic_v1", "run_id": "synthetic_run_001", "asset_id": "pump_asset_001", "generated_row_id": "synthetic_run_001_obs_000000000501", "observation": {"observation_index": 501, "observation_timestamp": "2026-04-16T08:20:00+00:00", "batch_id": 1, "row_in_batch": 500, "global_cycle_id": 501, "stream_state": "normal", "phase": "normal"}, "sensor": {"sensor_name": "sensor_00", "sensor_index": 0, "sensor_value": 2.4047780789116113, "message_sequence_index": 0}, "metadata": {"meta_episode_id": 0, "meta_primary_fault_type": "drift_down", "meta_magnitude": 1.2124996276063957, "created_at": "2026-05-24T23:54:01.146754+00:00"}, "telemetry": {"is_telemetry_event": false, "telemetry_event_type": null}, "producer": {"producer_send_attempt": 1, "queued_at": "2026-05-26T00:29:51.868662+00:00"}}


----

#### One Batch

In [8]:
# -----------------------------------------------------------------------------
# Optional smoke test: run exactly one producer batch
# -----------------------------------------------------------------------------
# This is not the normal official Stage 7 execution path.
# Use only when you want to test one batch before running the loop.

#RUN_ONE_BATCH_PRODUCER_SMOKE_TEST_FLAG = False

if RUN_ONE_BATCH_PRODUCER_SMOKE_TEST_FLAG:
    one_batch_result = run_send_queue_producer_once(
        engine=engine,
        dataset_id=DATASET_ID,
        run_id=RUN_ID,
        schema=SCHEMA,
        queue_table=SEND_QUEUE_TABLE_NAME,
        control_table=SIMULATION_TABLE_NAME,
        producer_worker_id=PRODUCER_WORKER_ID,
        client_id=CLIENT_ID,
        flush_timeout_seconds=FLUSH_TIMEOUT_SECONDS,
    )

    display(one_batch_result)
else:
    print("RUN_ONE_BATCH_PRODUCER_SMOKE_TEST_FLAG is False. Skipping one-batch smoke test.")

RUN_ONE_BATCH_PRODUCER_SMOKE_TEST_FLAG is False. Skipping one-batch smoke test.


----

In [9]:
post_send_queue_status_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        queue_status,
        COUNT(*) AS row_count,
        COUNT(*) FILTER (WHERE claim_token IS NULL) AS null_claim_token_count,
        COUNT(*) FILTER (WHERE claim_token IS NOT NULL) AS populated_claim_token_count,
        COUNT(*) FILTER (WHERE producer_sent_at IS NULL) AS unsent_count,
        COUNT(*) FILTER (WHERE producer_sent_at IS NOT NULL) AS sent_count,
        COUNT(*) FILTER (WHERE producer_delivery_error IS NOT NULL) AS error_count
    FROM "{SCHEMA}"."{SEND_QUEUE_TABLE_NAME}"
    WHERE dataset_id = :dataset_id
      AND run_id = :run_id
    GROUP BY queue_status
    ORDER BY row_count DESC
    """,
    params={
        "dataset_id": DATASET_ID,
        "run_id": RUN_ID,
    },
)

display(post_send_queue_status_dataframe)

,queue_status,row_count,null_claim_token_count,populated_claim_token_count,unsent_count,sent_count,error_count
0,pending,11674000,11674000,0,11674000,0,0
1,sent,26000,0,26000,0,26000,0


----

#### Loop

In [ ]:
# -----------------------------------------------------------------------------
# Run Kafka producer loop
# -----------------------------------------------------------------------------
# Official Stage 7 execution path.
#
# For first test:
#   RUN_PRODUCER_LOOP_FLAG = True
#   MAX_BATCHES = 1
#
# For full queue drain:
#   RUN_PRODUCER_LOOP_FLAG = True
#   MAX_BATCHES = None
#
# The loop stops when:
# - queue is empty
# - control row disables the run
# - max_batches is reached
# - a failure occurs and stop_on_failure=True

#RUN_PRODUCER_LOOP_FLAG = False

if RUN_PRODUCER_LOOP_FLAG:
    loop_results = run_send_queue_producer_loop(
        engine=engine,
        dataset_id=DATASET_ID,
        run_id=RUN_ID,
        schema=SCHEMA,
        queue_table=SEND_QUEUE_TABLE_NAME,
        control_table=SIMULATION_TABLE_NAME,
        producer_worker_id=PRODUCER_WORKER_ID,
        client_id=CLIENT_ID,
        max_batches=MAX_BATCHES,
        stop_on_failure=STOP_ON_FAILURE_FLAG,
        flush_timeout_seconds=FLUSH_TIMEOUT_SECONDS,
        enable_progress_logging=True,
        progress_every_batches=1,
    )

    display(loop_results)
else:
    print("RUN_PRODUCER_LOOP_FLAG is False. Skipping producer loop.")

#### Preview Payload Before Sending

In [ ]:
# Dispose Engine
engine.dispose()